In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"

In [2]:
import torch
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [3]:
model_path = 'models/xl/source'

In [4]:
seq_length = 512

from src.xl_wrapper import RuGPT3XL
model = RuGPT3XL.from_pretrained(
    "aiforever/rugpt3xl",
    weights_path=f"models/xl/source/mp_rank_00_model_states.pt",
    deepspeed_config_path="src/deepspeed_config/gpt3_xl_2048.json",
    #deepspeed_config_path="src/deepspeed_config/gpt3_xl_sparse_2048.json",
    seq_len=seq_length,
    #torch_dtype=torch.bfloat16
)
tokenizer = model.tokenizer

[2024-10-04 18:35:18,340] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
> initializing model parallel with size 1
[2024-10-04 18:35:20,375] [INFO] [config.py:733:__init__] Config mesh_device None world_size = 1


[W1004 18:35:20.682703294 socket.cpp:752] [c10d] The client socket cannot be initialized to connect to [localhost]:6000 (errno: 97 - Address family not supported by protocol).
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/workspaces/gpt/src/xl_wrapper.py:84: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weight

In [30]:
model.generate('раз два три четыре пять', early_stopping=True, max_length=20)



TypeError: repeat_interleave() received an invalid combination of arguments - got (NoneType, dim=int), but expected one of:
 * (Tensor repeats, int dim = None, *, int output_size = None)
 * (int repeats, int dim = None, *, int output_size = None)


In [5]:
from transformers import get_linear_schedule_with_warmup
model.cuda()
model.train()
lr = 1e-5
optimizer = torch.optim.AdamW(params=model.parameters(), lr=lr)

In [6]:
class TextDataset(Dataset):
    def __init__(self, path, tokenizer, seq_length=512):
        with open(path) as f:
            data = f.read()
        tokens = tokenizer.encode(data)
        examples = []
        for i in range(0, len(tokens) - seq_length + 1, seq_length):
            examples.append(tokens[i:i + seq_length])
        self.samples = torch.LongTensor(examples)
        print('Loaded samples:', len(self.samples))
    
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, item):
        return self.samples[item]

In [7]:
train_dataset = TextDataset('dataset/pelevin_train.txt', tokenizer)
valid_dataset = TextDataset('dataset/pelevin_valid.txt', tokenizer)
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=1, shuffle=False)

Loaded samples: 10502
Loaded samples: 255


In [8]:
tokenizer.decode(train_dataset[0])

'Annotation\n\n\nВбойщик KGBT+ (автор классических стримов «Катастрофа», «Летитбизм» и других) известен всей планете как титан перформанса и духа. Если вы не слышали его имени, значит, эпоха green power для вас еще не наступила и завоевавшее планету искусство B2B (brain-to-brain streaming) каким-то чудом обошло вас стороной.\n\nНо эта книга – не просто очередное жизнеописание звезды шоу-биза. Это учебник успеха. Великий вбойщик дает множество мемо-советов нацеленному на победу молодому исполнителю. KGBT+ подробно рассказывает историю создания своих шедевров и комментирует сложные факты своей биографии, включая убийства, покушения и почти вековую отсидку в баночной тюрьме, а также опровергает многочисленные слухи о своей личной жизни. Настоящее издание впервые включает повесть «Дом Бахии» о прошлой (предположительно) жизни легендарного вбойщика в Японии и Бирме.\n\nКнига не только подарит вам несколько интересных вечеров, но и познакомит с аутентичными древними психотехниками, применени

In [9]:
from torch import nn

In [10]:
criterion = nn.CrossEntropyLoss()

In [11]:
#loss = criterion(outputs['logits'][:, :-1].squeeze(), val_batch[:, 1:].cuda().squeeze());loss

NameError: name 'outputs' is not defined

In [13]:
#loss = criterion(outputs['logits'].squeeze(), val_batch.cuda().squeeze());loss

NameError: name 'outputs' is not defined

In [25]:
test = 'раз два три четыре пять'
tokens = torch.LongTensor(tokenizer.encode(test)).cuda()[None,]
outputs = model(input_ids=tokens, labels=tokens)
print(outputs.loss)#[0].item())

[tensor(11.3925, device='cuda:0', grad_fn=<NllLossBackward0>)]


In [16]:
outputs.loss[0].backward()

NameError: name 'outputs' is not defined

In [14]:
# Define the warmup steps
num_warmup_steps = 100
epochs = 1
# Calculate total training steps
num_training_steps = len(train_loader) * epochs  

# Create the learning rate scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

for epoch in range(epochs): 
    print('Epoch', epoch)

    progressbar = tqdm(train_loader)
    losses = []
    for step, batch in enumerate(progressbar):
        # Validation every 100 steps
        if step % 100 == 0:
            # Put the model in evaluation mode
            model.eval()

            val_losses = []
            with torch.no_grad():  # No need to calculate gradients during validation
                for val_batch in tqdm(valid_loader):
                    val_batch = val_batch.cuda()
                    outputs = model(input_ids=val_batch, labels=val_batch)
                    #, labels=val_batch
                    #val_loss = criterion(outputs['logits'][:, :-1].squeeze(), val_batch[:, 1:].cuda().squeeze())
                    val_losses.append(outputs.loss[0].item())

            avg_val_loss = np.mean(val_losses)
            print(f"Step {step}: Validation Loss: {avg_val_loss:.4f}")

        # Put the model back in training mode
        model.train()
        batch = batch.cuda()
        outputs = model(input_ids=batch, labels=batch)
        loss = outputs.loss
        print(loss)
        loss.backward()
        #torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.detach().item())
        progressbar.set_description(
            f"Loss: {np.mean(losses[-100:]):.3f}, LR: {scheduler.get_last_lr()[0]:.2e}"
        )
        # Update the learning rate
        scheduler.step()


Epoch 0


  0%|          | 0/10502 [00:04<?, ?it/s]

Step 0: Validation Loss: 11.1889
[tensor(11.1463, device='cuda:0', grad_fn=<NllLossBackward0>)]


AttributeError: 'list' object has no attribute 'backward'

In [12]:
output_path='candidates/'+str(lr).replace('-','')

In [13]:
lr

1e-05

In [ ]:
# Step 0: Validation Loss: 3.2091
# 1e-05 Loss: 2.9570

In [14]:
tokenizer.save_pretrained(output_path)
model.save_pretrained(output_path)

[2024-10-04 16:37:51,885] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)
